# Phase 3 — Reasoning Generalisation Evaluation (Standalone)

**Fully self-contained notebook.** No external code files needed.
You only need to provide:
- The 4 checkpoint `.pt` files (pretrained + fine-tuned, Hindi + Nepali)
- The 2 tokeniser `.model` files
- The 2 test JSONL files

**Edit Section 1 paths before running.**

## Section 1 — Configuration (edit paths here)

In [ ]:
# ================================================================
# CONFIGURATION — set these paths before running
# ================================================================

# ── Checkpoints ──────────────────────────────────────────────────
HINDI_PRETRAINED_CKPT  = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/hindi_checkpoints/step_00020000.pt"
NEPALI_PRETRAINED_CKPT = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/nepali_checkpoints/step_00020000.pt"
HINDI_FINETUNED_CKPT   = "/kaggle/working/results/phase3_finetune/hindi/step_00003000.pt"
NEPALI_FINETUNED_CKPT  = "/kaggle/working/results/phase3_finetune/nepali/step_00003000.pt"

# ── Tokenisers ───────────────────────────────────────────────────
HINDI_TOKENIZER  = "/kaggle/input/datasets/negative00/phase2-cloud-data/hindi/tokenizer/models/hindi_bpe_10k_final.model"
NEPALI_TOKENIZER = "/kaggle/input/datasets/negative00/phase2-cloud-data/nepali/tokenizer/models/nepali_bpe_10k_assignment.model"

# ── Test sets ────────────────────────────────────────────────────
HINDI_TEST  = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/hindi_test.jsonl"
NEPALI_TEST = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/nepali_test.jsonl"

# ── Output directory ─────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/results/phase3_finetune"

# ── Known-good reference values ──────────────────────────────────
KNOWN_RESULTS = {
    'hindi_pretrained':  {'correct': 40,  'total': 1200},
    'hindi_finetuned':   {'correct': 559, 'total': 1200},
    'nepali_pretrained': {'correct': 22,  'total': 1200},
    'nepali_finetuned':  {'correct': 495, 'total': 1200},
}

# ── Expected familiar/held-out counts ────────────────────────────
EXPECTED_SPLITS = {
    'hindi':  {'familiar': 621, 'held_out': 579},
    'nepali': {'familiar': 599, 'held_out': 601},
}

MAX_NEW_TOKENS = 20

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"Output dir : {OUTPUT_DIR}")


## Section 2 — Install & Environment

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece", "-q"], check=True)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")


## Section 3 — Model Architecture

In [ ]:
# ── LMConfig (from phase2/src/config.py) ─────────────────────────
from dataclasses import dataclass

@dataclass
class LMConfig:
    vocab_size: int = 10000
    d_model: int = 448
    n_layers: int = 8
    n_heads: int = 7
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    bias: bool = True

print("LMConfig defined.")


In [ ]:
# ── DecoderOnlyLM (from phase2/src/model.py) ─────────────────────
import math
import torch
import torch.nn as nn

class InputEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.max_seq_len = config.max_seq_len
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds max_seq_len {self.max_seq_len}")
        device = input_ids.device
        position_ids = torch.arange(0, T, dtype=torch.long, device=device)
        x = self.token_embedding(input_ids) + self.position_embedding(position_ids)
        return self.dropout(x)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        self.q_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(
            1, 1, config.max_seq_len, config.max_seq_len)
        self.register_buffer("bias_mask", mask)

    def forward(self, x, return_attention=False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias_mask[:, :, :T, :T] == 0, float('-inf'))
        att = torch.nn.functional.softmax(att, dim=-1)
        att_d = self.attn_dropout(att)
        y = (att_d @ v).transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        if return_attention:
            return y, att
        return y


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff, bias=config.bias),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_ff, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn  = FeedForward(config)

    def forward(self, x, return_attention=False):
        if return_attention:
            attn_out, att_weights = self.attn(self.ln_1(x), return_attention=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.ln_1(x))
        x = x + self.ffn(self.ln_2(x))
        if return_attention:
            return x, att_weights
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = InputEmbeddings(config)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f    = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)
        self.lm_head.weight = self.embedding.token_embedding.weight  # weight tying

    def forward(self, input_ids, return_attention=False):
        x = self.embedding(input_ids)
        attentions = [] if return_attention else None
        for block in self.blocks:
            if return_attention:
                x, att = block(x, return_attention=True)
                attentions.append(att)
            else:
                x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if return_attention:
            return logits, attentions
        return logits

print(f"DecoderOnlyLM defined. Params: {sum(p.numel() for p in DecoderOnlyLM(LMConfig()).parameters()):,}")


## Section 4 — Checkpoint Loader

In [ ]:
# ── load_checkpoint (from phase2/src/trainer.py) ──────────────────
import torch
from pathlib import Path
from typing import Optional

CHECKPOINT_VERSION = 1

def load_checkpoint(
    checkpoint_path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    device=None,
    restore_rng=True,
):
    if device is None:
        device = torch.device("cpu")
    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if payload.get("version") != CHECKPOINT_VERSION:
        raise ValueError(
            f"Checkpoint version mismatch. "
            f"Expected {CHECKPOINT_VERSION}, got {payload.get('version')}"
        )
    model.load_state_dict(payload["model_state"])
    if optimizer  is not None: optimizer.load_state_dict(payload["optimizer_state"])
    if scheduler  is not None: scheduler.load_state_dict(payload["scheduler_state"])
    if scaler is not None and "scaler_state" in payload:
        scaler.load_state_dict(payload["scaler_state"])
    return payload

print("load_checkpoint defined.")


## Section 5 — Evaluation Helpers

In [ ]:
# ── Evaluation helpers (from phase3/scripts/evaluate_reasoning.py) ─
import re
import json
import torch
import sentencepiece as spm
from collections import defaultdict

# Held-out templates (source of truth: validate_dataset.py)
UNSEEN_TEMPLATE_IDS = frozenset({
    'H3', 'H7', 'H9', 'H22', 'H31', 'H40',   # Hindi held-out
    'N3', 'N7', 'N9', 'N22', 'N31', 'N40',   # Nepali held-out
})

_UNK_SURFACE = '\u2047'  # ⁇

def is_held_out(example):
    """True iff the example uses at least one held-out template."""
    return any(t in UNSEEN_TEMPLATE_IDS for t in example.get('template_id', '').split('+'))

def extract_answer(raw_continuation):
    """
    Deterministic first-word extraction.
    Strips leading whitespace, extracts first non-whitespace/non-punct word.
    Returns '' if nothing extracted or if extracted token is UNK surface '⁇'.
    """
    text = raw_continuation.lstrip()
    match = re.match(r'^([^\s.,;!?\u0964]+)', text)
    if not match:
        return ''
    extracted = match.group(1)
    if extracted == _UNK_SURFACE:
        return ''
    return extracted

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def generate_answer(model, sp, prompt_text, max_new_tokens=20, device='cpu'):
    model.eval()
    prompt_ids = sp.Encode(prompt_text, out_type=int)
    input_ids  = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    eos_id     = sp.eos_id()
    generated  = []
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if input_ids.shape[1] > model.config.max_seq_len:
                input_ids = input_ids[:, -model.config.max_seq_len:]
            next_tok = model(input_ids)[0, -1, :].argmax().item()
            if next_tok == eos_id:
                break
            generated.append(next_tok)
            input_ids = torch.cat([input_ids, input_ids.new_tensor([[next_tok]])], dim=1)
    raw = sp.Decode(generated) if generated else ''
    return extract_answer(raw)

def run_eval(model, sp, dataset, device, max_new_tokens=20, label=''):
    buckets = {'overall': [0, 0], 'familiar': [0, 0], 'held_out': [0, 0]}
    for i, d in enumerate(dataset):
        prompt = f"\u092a\u094d\u0930\u0936\u094d\u0928: {d['premise']} {d['question']} \u0909\u0924\u094d\u0924\u0930: "
        gt   = str(d['answer']).strip()
        pred = generate_answer(model, sp, prompt, max_new_tokens=max_new_tokens, device=device)
        hit  = int(pred == gt)
        buckets['overall'][0] += hit;  buckets['overall'][1] += 1
        if is_held_out(d):
            buckets['held_out'][0] += hit; buckets['held_out'][1] += 1
        else:
            buckets['familiar'][0] += hit; buckets['familiar'][1] += 1
        if (i + 1) % 100 == 0:
            c, t = buckets['overall']
            print(f"  [{label}] {i+1}/{len(dataset)}  acc={c/t*100:.1f}%")
    def fmt(pair):
        c, t = pair
        return {'correct': c, 'total': t, 'accuracy': c / t if t else 0.0}
    return {k: fmt(v) for k, v in buckets.items()}

def load_model_from_ckpt(ckpt_path, device):
    model = DecoderOnlyLM(LMConfig()).to(device)
    load_checkpoint(ckpt_path, model, device=torch.device(device), restore_rng=False)
    model.eval()
    return model

print("Evaluation helpers defined.")
print(f"UNSEEN_TEMPLATE_IDS = {sorted(UNSEEN_TEMPLATE_IDS)}")


## Section 6 — Test-Set Integrity Check

In [ ]:
# ── Test-set integrity check ─────────────────────────────────────
test_data = {}
for lang, path in [('hindi', HINDI_TEST), ('nepali', NEPALI_TEST)]:
    ds   = load_jsonl(path)
    fam  = sum(1 for d in ds if not is_held_out(d))
    held = sum(1 for d in ds if     is_held_out(d))
    exp  = EXPECTED_SPLITS[lang]
    ok   = (fam == exp['familiar'] and held == exp['held_out'])
    status = '✓' if ok else 'MISMATCH'
    print(f"{lang}: total={len(ds)}  familiar={fam} (expected {exp['familiar']})  "
          f"held-out={held} (expected {exp['held_out']})  [{status}]")
    if not ok:
        raise AssertionError(
            f"{lang} familiar/held-out counts do not match expected values.\n"
            f"Check that HINDI_TEST / NEPALI_TEST point to the correct files."
        )
    test_data[lang] = ds
print("\nIntegrity check PASSED.")


## Section 7 — Run All Four Evaluations

In [ ]:
# ── Run all four evaluations ─────────────────────────────────────
import gc

device = DEVICE
results = {}

RUNS = [
    ('hindi',  'pretrained', HINDI_PRETRAINED_CKPT,  HINDI_TOKENIZER),
    ('hindi',  'finetuned',  HINDI_FINETUNED_CKPT,   HINDI_TOKENIZER),
    ('nepali', 'pretrained', NEPALI_PRETRAINED_CKPT,  NEPALI_TOKENIZER),
    ('nepali', 'finetuned',  NEPALI_FINETUNED_CKPT,   NEPALI_TOKENIZER),
]

for lang, model_type, ckpt_path, tok_path in RUNS:
    key = f"{lang}_{model_type}"
    print(f"\n{'='*60}")
    print(f"Evaluating : {key}")
    print(f"Checkpoint : {ckpt_path}")

    if not Path(ckpt_path).exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    sp = spm.SentencePieceProcessor()
    sp.Load(tok_path)
    assert sp.GetPieceSize() == 10000, f"Expected vocab 10000, got {sp.GetPieceSize()}"

    model = load_model_from_ckpt(ckpt_path, device)
    res   = run_eval(model, sp, test_data[lang], device, max_new_tokens=MAX_NEW_TOKENS, label=key)
    results[key] = res

    print(f"  Overall  : {res['overall']['correct']}/{res['overall']['total']} = {res['overall']['accuracy']*100:.2f}%")
    print(f"  Familiar : {res['familiar']['correct']}/{res['familiar']['total']} = {res['familiar']['accuracy']*100:.2f}%")
    print(f"  Held-out : {res['held_out']['correct']}/{res['held_out']['total']} = {res['held_out']['accuracy']*100:.2f}%")

    ref = KNOWN_RESULTS.get(key)
    if ref:
        if res['overall']['correct'] != ref['correct']:
            print(f"  ⚠ DISCREPANCY: expected {ref['correct']} correct, got {res['overall']['correct']}")
        else:
            print(f"  ✓ Overall correct matches reference ({ref['correct']})")

    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\nAll evaluations complete.")


## Section 8 — Results Tables & Save

In [ ]:
# ── Results tables & save ─────────────────────────────────────────
import csv
from pathlib import Path

def pct(acc): return f"{acc*100:.2f}%"
def pp(a, b): return f"+{(b-a)*100:.2f} pp" if b >= a else f"{(b-a)*100:.2f} pp"

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(f"{'Language':<10} {'Model':<14} {'Overall':>10} {'Familiar':>10} {'Held-out':>10}")
print("-" * 54)

rows, improvement_rows = [], []
for lang in ['hindi', 'nepali']:
    for model_type in ['pretrained', 'finetuned']:
        key = f"{lang}_{model_type}"
        r   = results[key]
        row = {
            'language': lang, 'model': model_type,
            'overall_correct':  r['overall']['correct'],
            'overall_total':    r['overall']['total'],
            'overall_acc':      r['overall']['accuracy'],
            'familiar_correct': r['familiar']['correct'],
            'familiar_total':   r['familiar']['total'],
            'familiar_acc':     r['familiar']['accuracy'],
            'heldout_correct':  r['held_out']['correct'],
            'heldout_total':    r['held_out']['total'],
            'heldout_acc':      r['held_out']['accuracy'],
        }
        rows.append(row)
        print(f"{lang:<10} {model_type:<14} {pct(row['overall_acc']):>10} "
              f"{pct(row['familiar_acc']):>10} {pct(row['heldout_acc']):>10}")

print("\n" + "="*80)
print("IMPROVEMENT (fine-tuned vs pretrained)")
print("="*80)
print(f"{'Language':<10} {'Overall Δ':>12} {'Familiar Δ':>12} {'Held-out Δ':>12}")
print("-" * 46)

for lang in ['hindi', 'nepali']:
    pre = results[f"{lang}_pretrained"]
    ft  = results[f"{lang}_finetuned"]
    delta_overall  = ft['overall']['accuracy']  - pre['overall']['accuracy']
    delta_familiar = ft['familiar']['accuracy'] - pre['familiar']['accuracy']
    delta_heldout  = ft['held_out']['accuracy'] - pre['held_out']['accuracy']
    improvement_rows.append({
        'language': lang,
        'delta_overall_pp':  round(delta_overall  * 100, 2),
        'delta_familiar_pp': round(delta_familiar * 100, 2),
        'delta_heldout_pp':  round(delta_heldout  * 100, 2),
    })
    print(f"{lang:<10} {pp(pre['overall']['accuracy'],  ft['overall']['accuracy']):>12} "
          f"{pp(pre['familiar']['accuracy'], ft['familiar']['accuracy']):>12} "
          f"{pp(pre['held_out']['accuracy'], ft['held_out']['accuracy']):>12}")

# Save CSV
csv_path = Path(OUTPUT_DIR) / 'reasoning_generalization_comparison.csv'
fieldnames = ['language','model',
              'overall_correct','overall_total','overall_acc',
              'familiar_correct','familiar_total','familiar_acc',
              'heldout_correct','heldout_total','heldout_acc']
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader(); w.writerows(rows)
print(f"\nCSV saved: {csv_path}")

# Save Markdown
md_path = Path(OUTPUT_DIR) / 'reasoning_generalization_summary.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('# Phase 3 Reasoning Generalisation Summary\n\n')
    f.write('## Comparison (Overall / Familiar / Held-out accuracy)\n\n')
    f.write('| Language | Model | Overall | Familiar | Held-out |\n')
    f.write('|---|---|---|---|---|\n')
    for r in rows:
        f.write(f"| {r['language']} | {r['model']} "
                f"| {r['overall_correct']}/{r['overall_total']} ({r['overall_acc']*100:.2f}%) "
                f"| {r['familiar_correct']}/{r['familiar_total']} ({r['familiar_acc']*100:.2f}%) "
                f"| {r['heldout_correct']}/{r['heldout_total']} ({r['heldout_acc']*100:.2f}%) |\n")
    f.write('\n## Improvement (fine-tuned vs pretrained, pp)\n\n')
    f.write('| Language | Overall Δ | Familiar Δ | Held-out Δ |\n')
    f.write('|---|---|---|---|\n')
    for ir in improvement_rows:
        sign = lambda x: f'+{x:.2f}' if x >= 0 else f'{x:.2f}'
        f.write(f"| {ir['language']} "
                f"| {sign(ir['delta_overall_pp'])} pp "
                f"| {sign(ir['delta_familiar_pp'])} pp "
                f"| {sign(ir['delta_heldout_pp'])} pp |\n")
    f.write(f'\n*Held-out templates: {sorted(UNSEEN_TEMPLATE_IDS)}*\n')
print(f"Markdown saved: {md_path}")


## Done

Results written to:
- `results/phase3_finetune/reasoning_generalization_comparison.csv`
- `results/phase3_finetune/reasoning_generalization_summary.md`